Load the table into a pandas dataframe.

In [39]:
import pandas
import sqlite3

database_connection = sqlite3.connect('baseball_db.sqlite')
dataset = pandas.read_sql_query("SELECT * FROM model_dataset", database_connection)

database_connection.close()

print(dataset.head())

   mlb_ID  seasons_bat  career_PA  avg_offense  total_bat_war  peak_bat_war  \
0  110001           23      13941    33.465217         143.07          9.44   
1  110002            7       1045    -7.342857          -2.77          0.92   
2  110003           13          5    -0.046154          -0.07          0.00   
3  110004            1         49    -0.750000          -0.06          0.01   
4  110005            9       3478     2.570000           8.72          3.63   

   seasons_pitch  career_GS  avg_era_plus  total_pitch_war  peak_pitch_war  \
0              0          0      0.000000             0.00            0.00   
1              0          0      0.000000             0.00            0.00   
2             13         91    126.585193            15.09            2.54   
3              0          0      0.000000             0.00            0.00   
4              0          0      0.000000             0.00            0.00   

   hof_indicator  
0              1  
1              0  

Seperate features from target and train/test split

In [40]:
from sklearn.model_selection import train_test_split

X = dataset.drop(['mlb_ID', 'hof_indicator'], axis=1)
y = dataset['hof_indicator']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Correct imbalance with SMOTE

In [41]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

Train the model

In [42]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(class_weight='balanced')
model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced')

Evaluate

In [43]:
from sklearn.metrics import classification_report, roc_auc_score

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))
print("AUC-ROC:", roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]))

              precision    recall  f1-score   support

           0       1.00      0.99      0.99      4185
           1       0.23      0.56      0.33        25

    accuracy                           0.99      4210
   macro avg       0.61      0.77      0.66      4210
weighted avg       0.99      0.99      0.99      4210

AUC-ROC: 0.9791875746714457


Test catboost model

In [44]:
from catboost import CatBoostClassifier

catboost_model = CatBoostClassifier(verbose=0, random_seed=42)
catboost_model.fit(X_train, y_train)

Evaluate

In [45]:
y_pred_catboost = catboost_model.predict(X_test)
print("CatBoost Classification Report:")
print(classification_report(y_test, y_pred_catboost))
print("CatBoost AUC-ROC:", roc_auc_score(y_test, catboost_model.predict_proba(X_test)[:, 1]))

CatBoost Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.98      0.99      4185
           1       0.20      0.68      0.31        25

    accuracy                           0.98      4210
   macro avg       0.60      0.83      0.65      4210
weighted avg       0.99      0.98      0.99      4210

CatBoost AUC-ROC: 0.953290322580645
